In [1]:
import pandas as pd
import numpy as np

# -----------------------------
# PARAMETERS
# -----------------------------
n_rows = 1000   # number of synthetic records
output_csv = "univseal_monster_boxsets_sales.csv"

# possible values for categorical fields
formats = ["Digital", "DVD", "Blu-ray", "4K UHD"]
regions = ["North America", "Europe", "Asia", "Latin America", "Australia"]
age_groups = ["18-24", "25-34", "35-44", "45-54", "55+"]
genders = ["Male", "Female", "Non-binary", "Prefer not to say"]
income_brackets = ["<30k", "30-60k", "60-100k", "100-150k", "150k+"]

# -----------------------------
# SYNTHETIC DATA GENERATION
# -----------------------------
rng = np.random.default_rng(seed=42)

# 1) Basic sale info
sale_id = np.arange(1, n_rows + 1)
sale_dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    rng.integers(0, 365, size=n_rows), unit="D"
)
chosen_format = rng.choice(formats, size=n_rows, p=[0.4, 0.25, 0.25, 0.10])  # example weights
region = rng.choice(regions, size=n_rows)

# 2) Units sold and pricing
# Assume units sold per transaction mostly 1-3, with occasional bundle orders
units_sold = rng.integers(1, 4, size=n_rows)

# Example base list price by format
base_price_map = {
    "Digital": 19.99,
    "DVD": 24.99,
    "Blu-ray": 29.99,
    "4K UHD": 39.99,
}
unit_price = np.array([base_price_map[f] for f in chosen_format])

# Some random discount or markup, e.g. -20% to +5%
price_multiplier = rng.uniform(0.8, 1.05, size=n_rows)
unit_price = np.round(unit_price * price_multiplier, 2)

# Revenue
revenue = np.round(unit_price * units_sold, 2)

# 3) Cost per unit (to estimate profit)
# Example cost scenarios: digital cost low, physical cost higher.
# We'll add some randomness but keep cost < price.
cost_base = {
    "Digital": 5.00,
    "DVD": 12.00,
    "Blu-ray": 15.00,
    "4K UHD": 20.00,
}
cost_variation = rng.uniform(0.9, 1.1, size=n_rows)
cost_per_unit = np.round(
    np.array([cost_base[f] for f in chosen_format]) * cost_variation, 2
)

# Total cost for the transaction
total_cost = np.round(cost_per_unit * units_sold, 2)

# 4) Profit and profit margin
profit = np.round(revenue - total_cost, 2)
# Avoid divide-by-zero if revenue is 0; here revenue > 0 by design
profit_margin = np.round(np.where(revenue > 0, profit / revenue, 0), 4)

# 5) Demographics
age_group = rng.choice(age_groups, size=n_rows, p=[0.1, 0.3, 0.25, 0.2, 0.15])
gender = rng.choice(genders, size=n_rows, p=[0.45, 0.45, 0.05, 0.05])
income_bracket = rng.choice(income_brackets, size=n_rows, p=[0.2, 0.3, 0.25, 0.15, 0.1])

# -----------------------------
# BUILD DATAFRAME
# -----------------------------
df = pd.DataFrame({
    "sale_id": sale_id,
    "sale_date": sale_dates,
    "region": region,
    "format": chosen_format,
    "units_sold": units_sold,
    "unit_price": unit_price,
    "revenue": revenue,
    "cost_per_unit": cost_per_unit,
    "total_cost": total_cost,
    "profit": profit,
    "profit_margin": profit_margin,  # as a proportion, e.g. 0.25 = 25%
    "customer_age_group": age_group,
    "customer_gender": gender,
    "customer_income_bracket": income_bracket
})

# Optional: convert profit_margin to percentage string for readability
df["profit_margin_pct"] = (df["profit_margin"] * 100).round(2).astype(str) + "%"

# -----------------------------
# SAVE TO CSV
# -----------------------------
df.to_csv(output_csv, index=False)

print(f"Generated {n_rows} rows and saved to {output_csv}")

Generated 1000 rows and saved to univseal_monster_boxsets_sales.csv
